# PhishGuard: 02. Feature Engineering & Static URL Analysis

## Research Purpose
> **Objective:** Transform raw URL strings into robust numerical and structural indicators without making network requests or executing untrusted code.

This notebook demonstrates `extract_url_features(url)` across various attack patterns and computes statistical distributions of features between legitimate and phishing URLs.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_extractor import extract_url_features, FEATURE_NAMES

print(f"Feature extractor loaded. Total extracted features: {len(FEATURE_NAMES)}")

## 1. Single URL Extraction Examples
We test several canonical URL patterns:
1. **Legitimate Institutional URL** (`https://www.cam.ac.uk`)
2. **IP-Based Phishing URL** (`http://192.168.1.1/admin/login`)
3. **Social Engineering Keyword URL** (`http://secure-paypal-verify-account.update-login.com`)
4. **Credential Obfuscation URL** (`http://legit.com@phishing-target.com/signin`)
5. **Shortened URL** (`https://bit.ly/3xY9AbZ`)

In [ ]:
test_urls = [
    "https://www.cam.ac.uk",
    "http://192.168.1.1/admin/login",
    "http://secure-paypal-verify-account.update-login.com",
    "http://legit.com@phishing-target.com/signin",
    "https://bit.ly/3xY9AbZ"
]

records = [extract_url_features(u) for u in test_urls]
df_sample = pd.DataFrame(records, index=test_urls)
df_sample[["url_length", "has_https", "has_ip_address", "has_at_symbol", "has_url_shortener", "num_subdomains", "suspicious_keyword_count"]]

## 2. Statistical Analysis on PhiUSIIL Sample
To analyze statistical separation, we load a balanced sample of 2,000 legitimate and 2,000 phishing URLs from the raw PhiUSIIL dataset and extract features dynamically.

In [ ]:
raw_csv = PROJECT_ROOT / "data" / "raw" / "PhiUSIIL_Phishing_URL_Dataset.csv"
df_raw = pd.read_csv(raw_csv, usecols=["URL", "label"])

# Sample 2,000 benign (raw label 1) and 2,000 phishing (raw label 0)
sample_benign = df_raw[df_raw["label"] == 1].sample(n=2000, random_state=42)
sample_phish = df_raw[df_raw["label"] == 0].sample(n=2000, random_state=42)

df_eval = pd.concat([sample_benign, sample_phish]).reset_index(drop=True)
df_eval["target"] = df_eval["label"].apply(lambda x: 1 if x == 0 else 0)  # 1 = Phishing, 0 = Benign

# Dynamically extract features
extracted_rows = [extract_url_features(u) for u in df_eval["URL"]]
df_features = pd.DataFrame(extracted_rows)
df_features["target"] = df_eval["target"]

print(f"Extracted feature matrix shape: {df_features.shape}")

## 3. Comparative Feature Distributions
We evaluate how key indicators diverge between Legitimate and Phishing URLs.

In [ ]:
summary = df_features.groupby("target")[[
    "url_length", "has_https", "has_ip_address", "num_subdomains",
    "num_dots", "suspicious_keyword_present", "special_char_ratio"
]].mean().T

summary.columns = ["Legitimate (Mean)", "Phishing (Mean)"]
summary["Difference"] = summary["Phishing (Mean)"] - summary["Legitimate (Mean)"]
summary